this is the main code file for cARd detection

In [1]:
!pip install ultralytics opencv-python pillow torchvision torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.8 MB/s eta 0:00:00


In [2]:
import torch
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO
from torchvision import transforms
import torch.nn.functional as F

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
card_detector = YOLO("CardPicModel1.pt")          # detects full cards
symbol_digit_detector = YOLO("SymbolDigitModel2.pt")  # detects symbol + digit

In [4]:
class SymbolCNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3,32,3,padding=1)
        self.conv2 = torch.nn.Conv2d(32,64,3,padding=1)
        self.pool = torch.nn.MaxPool2d(2,2)
        self.fc1 = torch.nn.Linear(64*8*8,128)
        self.fc2 = torch.nn.Linear(128,4)

    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1,64*8*8)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

symbol_model = SymbolCNN()
symbol_model.load_state_dict(torch.load("symbol_classifier.pth", map_location="cpu"))
symbol_model.eval()

symbol_classes = ['C','D','H','S']

In [5]:
class RankCNN(torch.nn.Module):
    def __init__(self, n):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3,32,3,padding=1)
        self.conv2 = torch.nn.Conv2d(32,64,3,padding=1)
        self.pool = torch.nn.MaxPool2d(2,2)
        self.fc1 = torch.nn.Linear(64*8*8,128)
        self.fc2 = torch.nn.Linear(128,n)

    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1,64*8*8)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

with open("rank_classes.txt") as f:
    rank_classes = [l.strip() for l in f]

rank_model = RankCNN(len(rank_classes))
rank_model.load_state_dict(torch.load("rank_classifier.pth", map_location="cpu"))
rank_model.eval()

RankCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=13, bias=True)
)

In [6]:
transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

In [7]:
def predict_symbol(img):
    x = transform(img).unsqueeze(0)
    with torch.no_grad():
        _, i = torch.max(symbol_model(x),1)
    return symbol_classes[i.item()]

def predict_rank(img):
    x = transform(img).unsqueeze(0)
    with torch.no_grad():
        _, i = torch.max(rank_model(x),1)
    return rank_classes[i.item()]

In [8]:
def detect_cards(image_path):
    image = cv2.imread(image_path)
    results = []

    # 1️⃣ Detect full cards
    card_preds = card_detector(image)[0]

    for card_box in card_preds.boxes.xyxy:
        x1,y1,x2,y2 = map(int, card_box)
        card_crop = image[y1:y2, x1:x2]

        # 2️⃣ Detect symbol + digit
        sd_preds = symbol_digit_detector(card_crop)[0]

        symbol_img = None
        rank_img = None

        for box, cls in zip(sd_preds.boxes.xyxy, sd_preds.boxes.cls):
            bx1,by1,bx2,by2 = map(int, box)
            crop = card_crop[by1:by2, bx1:bx2]
            pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

            if int(cls) == 0:      # symbol class
                symbol_img = pil_crop
            else:                 # digit class
                rank_img = pil_crop

        if symbol_img and rank_img:
            rank = predict_rank(rank_img)
            symbol = predict_symbol(symbol_img)
            results.append(f"{rank}{symbol}")

    return results

In [9]:
cards = detect_cards("cards.jpg")
print(cards)


0: 640x384 13 cards, 47.9ms
Speed: 8.2ms preprocess, 47.9ms inference, 37.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x320 1 symbol, 1 digit, 59.1ms
Speed: 1.3ms preprocess, 59.1ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 320)

0: 640x352 1 symbol, 1 digit, 46.2ms
Speed: 1.6ms preprocess, 46.2ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 352)

0: 384x640 1 symbol, 1 digit, 38.3ms
Speed: 2.8ms preprocess, 38.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 416x640 1 symbol, 1 digit, 40.6ms
Speed: 2.4ms preprocess, 40.6ms inference, 1.2ms postprocess per image at shape (1, 3, 416, 640)

0: 288x640 1 symbol, 1 digit, 39.1ms
Speed: 1.4ms preprocess, 39.1ms inference, 1.3ms postprocess per image at shape (1, 3, 288, 640)

0: 640x384 1 symbol, 1 digit, 29.0ms
Speed: 2.2ms preprocess, 29.0ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x256 1 symbol, 1 digit, 37.3ms
Speed: 1.4ms preprocess, 